# ⚡️ SISR Surveillance — Colab Training Notebook

Run SRCNN/ESPCN super-resolution experiments (E1–E8) on Colab's free GPU (T4), then download the trained models back to your local machine.

## What this notebook does
1. Mounts Google Drive (persistent storage for checkpoints)
2. Installs PyTorch + dependencies
3. Unzips your project (uploaded to Drive) or clones from GitHub
4. Generates Set5/Set14 training pairs
5. Runs experiments on Colab GPU and saves checkpoints
6. Zips & downloads trained models to your local machine

## Before you start
- **Runtime → Change runtime type → GPU** (select T4)
- Upload your project folder as `project.zip` to Google Drive at:
  `MyDrive/cv_project/project.zip`
  — or push to GitHub and uncomment the `git clone` line in the Setup cell

---


In [ ]:
import torch, os, sys
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive mounted at /content/drive")


## Step 1: Install dependencies

Installs PyTorch, torchvision, OpenCV, scikit-image, tqdm, pyyaml, numpy, pillow, matplotlib.

In [ ]:
!pip install -q torch torchvision opencv-python-headless scikit-image tqdm pyyaml numpy pillow matplotlib
print("Dependencies installed.")


## Step 2: Set up the project workspace

Choose one of the three options below. The notebook will use whichever source it finds first.

**Option A:** Upload your project as `project.zip` to Google Drive at `MyDrive/cv_project/project.zip`
**Option B:** Push to GitHub and uncomment the `git clone` line
**Option C:** If files already exist in `/content/cv`, they are used as-is

In [ ]:
import os, zipfile, glob, shutil

WORK = '/content/cv'
DRIVE_PROJECT = '/content/drive/MyDrive/cv_project'
DRIVE_OUTPUTS = '/content/drive/MyDrive/cv_outputs'

os.makedirs(WORK, exist_ok=True)
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

# Option A: unzip project from Google Drive
zips = glob.glob(os.path.join(DRIVE_PROJECT, '*.zip'))
if zips:
    zip_path = zips[0]
    print(f"Found project zip: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(WORK)
    print(f"Project extracted to {WORK}")
elif os.path.exists(os.path.join(WORK, 'train', 'train_all.py')):
    print(f"Project already present at {WORK}")
else:
    print("""
    No project found. Please do ONE of:
    1. Upload your project folder as project.zip to Google Drive at:
       MyDrive/cv_project/project.zip
    2. Push to GitHub and uncomment the git clone line below:
    """)
    # Uncomment and edit the next line to clone from GitHub:
    # !git clone https://github.com/YOUR_USERNAME/cv.git {WORK}


## Step 3: Generate / prepare training data

Runs `data/download_sets.py` (creates synthetic Set5/Set14) and `data/prepare_pairs.py` (builds LR/HR pairs at 2× and 4×).

If you have real surveillance images, upload them to Drive and point `data/download_real_sets.py` / `data/prepare_real_pairs.py` at them instead.

In [ ]:
!cd /content/cv && python data/download_sets.py && python data/prepare_pairs.py
print("\n--- Prepared data tree ---")
!find /content/cv/data/pairs -maxdepth 2 -type d | sort


## Step 4: Select experiments to run

Default: E1–E4 (MSE loss, SRCNN & ESPCN at 2× and 4×).
Set `RUN_ALL = True` to run E1–E8 including perceptual-loss and degraded-training experiments.

Note: E5/E6/E8 use VGG11 perceptual loss (~500 MB weight download on first run).

In [ ]:
RUN_ALL = False  # set True to run E1-E8

if RUN_ALL:
    EXPS = ['E1','E2','E3','E4','E5','E6','E7','E8']
else:
    EXPS = ['E1','E2','E3','E4']

print(f"Experiments to run: {EXPS}")
print(f"Run all (incl. perceptual/degraded): {RUN_ALL}")


## Step 5: Train experiments on Colab GPU

Streams training output live. Each experiment saves its best checkpoint to:
`/content/cv/outputs/checkpoints/{model}_{scale}x_{loss}.pt`

E7/E8 automatically use `--degrade` (on-the-fly blur/noise/JPEG degradation).

In [ ]:
import subprocess, sys, os, time

WORK = '/content/cv'
CONFIG = 'configs/experiments.yaml'

degrade_exps = {'E7', 'E8'}

for exp in EXPS:
    print(f"\n{'='*70}")
    print(f"EXPERIMENT {exp}")
    print(f"{'='*70}\n")
    
    cmd = [
        sys.executable,
        'train/train_all.py',
        '--config', CONFIG,
        '--exp', exp,
    ]
    if exp in degrade_exps:
        cmd += ['--degrade', '--blur-ksize', '5', '--blur-sigma', '1.0', '--noise-sigma', '5.0', '--jpeg-quality', '75']
    
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=WORK,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    
    for line in proc.stdout:
        print(line, end='')
    
    proc.wait()
    dt = time.time() - t0
    
    if proc.returncode != 0:
        print(f"\n❌ Experiment {exp} FAILED (exit code {proc.returncode}) after {dt:.1f}s")
        break
    else:
        print(f"\n✅ Experiment {exp} COMPLETE in {dt:.1f}s")

print("\n\n=== All selected experiments finished ===")


## Step 6: Save checkpoints to Google Drive

Copies all trained models from `/content/cv/outputs/checkpoints/` to Google Drive so they persist after the Colab session ends.

In [ ]:
import shutil, os, glob

WORK = '/content/cv'
DRIVE_OUTPUTS = '/content/drive/MyDrive/cv_outputs'
DRIVE_CKPTS = os.path.join(DRIVE_OUTPUTS, 'checkpoints')
os.makedirs(DRIVE_CKPTS, exist_ok=True)

src_ckpts = os.path.join(WORK, 'outputs', 'checkpoints')
if os.path.isdir(src_ckpts):
    files = [f for f in os.listdir(src_ckpts) if f.endswith('.pt')]
    for f in files:
        shutil.copy2(os.path.join(src_ckpts, f), os.path.join(DRIVE_CKPTS, f))
    print(f"Copied {len(files)} checkpoint(s) to Google Drive:")
    for f in sorted(files):
        print(f"   {DRIVE_CKPTS}/{f}")
else:
    print("No checkpoints found at", src_ckpts)

print("\nDrive outputs folder:", DRIVE_OUTPUTS)


## Step 7: Download trained models back to your local machine

Creates a zip of everything in `/content/cv/outputs/` and triggers a browser download.

After downloading, unzip on your local machine at `C:\Users\ASUS\Desktop\cv\` to merge the trained models and logs into your project.

In [ ]:
import zipfile, os
from google.colab import files

WORK = '/content/cv'
OUTPUTS_DIR = os.path.join(WORK, 'outputs')
ZIP_PATH = '/content/cv_trained_models.zip'

if os.path.isdir(OUTPUTS_DIR) and os.listdir(OUTPUTS_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, dirs, files_ in os.walk(OUTPUTS_DIR):
            for f in files_:
                full = os.path.join(root, f)
                rel = os.path.relpath(full, WORK)
                z.write(full, rel)
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"Created {ZIP_PATH} ({size_mb:.1f} MB)")
    print("\nStarting download... (save this file to your local machine)")
    files.download(ZIP_PATH)
else:
    print("No outputs directory found. Training may not have run yet.")


## Step 8: Verify what came back

List the checkpoints that were saved to Google Drive (so you can re-download individually if needed).

In [ ]:
import os
DRIVE_CKPTS = '/content/drive/MyDrive/cv_outputs/checkpoints'
if os.path.isdir(DRIVE_CKPTS):
    print(f"Checkpoints on Google Drive ({DRIVE_CKPTS}):")
    for f in sorted(os.listdir(DRIVE_CKPTS)):
        path = os.path.join(DRIVE_CKPTS, f)
        size_kb = os.path.getsize(path) / 1024
        print(f"   {f}  ({size_kb:.1f} KB)")
else:
    print("No checkpoints folder on Drive yet.")

print("\nTo get models back to your local machine:")
print("  1. Download cv_trained_models.zip (from Step 7), OR")
print("  2. Download individual .pt files from Google Drive at MyDrive/cv_outputs/checkpoints/")
print("  3. Place them in C:\\Users\\ASUS\\Desktop\\cv\\outputs\\checkpoints\\")


---
## Quick reference: experiment matrix

| Exp | Model | Scale | Loss | Train → Test | Notes |
|-----|-------|-------|------|---------------|-------|
| E1 | SRCNN | 2× | MSE | Set5 → Set14 | |
| E2 | SRCNN | 4× | MSE | Set5 → Set14 | |
| E3 | ESPCN | 2× | MSE | Set5 → Set14 | |
| E4 | ESPCN | 4× | MSE | Set5 → Set14 | |
| E5 | ESPCN | 2× | MSE+Perceptual | Set5 → Set14 | Colab T4 recommended |
| E6 | ESPCN | 4× | MSE+Perceptual | Set5 → Set14 | Colab T4 recommended |
| E7 | ESPCN | 4× | MSE | Set5_degraded → Set14 | Uses `--degrade` |
| E8 | ESPCN | 4× | MSE+Perceptual | Set5_degraded → Set14 | Uses `--degrade`, Colab T4 |

---
*Generated for the SISR surveillance project at `C:\Users\ASUS\Desktop\cv`.*
